In [103]:
import pandas as pd

In [104]:
df = pd.read_json("raw_deals.json")

In [105]:
df["date"] = df["date"].str.replace("t/m", "")

In [106]:
df = df.rename(columns = {'date':'start_date'})

In [107]:
df["end_date"] = ""

In [108]:
df["end_date"] = df["start_date"].str[6:]

In [109]:
df["start_date"] = df["start_date"].str[:6]

In [110]:
df["discount"] = df["discount"].apply(lambda x: " ".join(x) if isinstance(x, list) else str(x))

In [111]:
# Keep everything that DOES NOT contain "bezorging" so no delivery!
df = df[~df["discount"].str.contains("bezorging", case=False, na=False)]

In [112]:
# Get rid of every product which is Online only!
df = df[~df["discount"].str.contains("Online", case=False, na=False)]

In [113]:
df["discount"] = df["discount"].str.split(",", n=1).str[1].str.strip()

In [114]:
import re

def keep_parts_with_numbers(text):
    if not isinstance(text, str):
        return text
    
    parts = text.split(",")
    filtered = []
    
    for part in parts:
        part = part.strip()
        has_number = re.search(r'\d', part)
        has_muv = re.search(r'M\.u\.v', part)
        has_only_winkel = re.search(r'Alleen in de winkel', part)
        if has_number and not has_muv:
            if not has_only_winkel:
                filtered.append(part)
    
    result = ", ".join(filtered)
    return result

df["discount"] = df["discount"].apply(keep_parts_with_numbers)

In [115]:
df["product_info"] = ""

In [116]:
discount_keys = ['voor', 'korting', 'gratis', 'prijs', 'halve', '%', '€', 'van', 'per']

def clean_and_sort(row):
    parts = str(row['discount']).split(',') 
    clean_discounts = []
    moving_to_info = []
    pattern_stuk_price = r'\d+\s*stuk.*\d'
    
    for p in parts:
        p = p.strip()
        if not p: continue 
        has_discount_word = any(word in p.lower() for word in discount_keys)
        has_stuk_with_price = bool(re.search(pattern_stuk_price, p, re.IGNORECASE))
        
        if has_discount_word or has_stuk_with_price:
            clean_discounts.append(p)
        else:
            moving_to_info.append(p)
        
    return pd.Series([", ".join(clean_discounts), ", ".join(moving_to_info)])

df[['discount', 'temp_extra']] = df.apply(clean_and_sort, axis=1)
df['product_info'] = df['product_info'].fillna('') + " " + df['temp_extra']
df['product_info'] = df['product_info'].str.strip() 
df = df.drop(columns=['temp_extra'])

In [117]:
from datetime import datetime
dutch_months = {
    "jan": "01", "feb": "02", "mrt": "03", "apr": "04",
    "mei": "05", "jun": "06", "jul": "07", "aug": "08",
    "sep": "09", "okt": "10", "nov": "11", "dec": "12"
}

def convert_dutch_date(date_str):
    if not isinstance(date_str, str):
        return None
    
    parts = date_str.strip().split(" ")
    day = int(parts[0])
    month = int(dutch_months[parts[1].lower()])
    
    current = datetime.now()
    year = current.year
    
    if month < current.month:
        year += 1
    
    return pd.to_datetime(f"{day:02d}-{month:02d}-{year}", format="%d-%m-%Y")

df["start_date"] = df["start_date"].apply(convert_dutch_date)
df["end_date"] = df["end_date"].apply(convert_dutch_date)

In [118]:
df["discounted_price"] = ""

In [119]:
pattern = r'\d+\s+stuks?\s+[\d.%]+'

# 1. Extract all matches into a new column (this creates the temporary list)
df['discount_split'] = df['discount'].str.findall(pattern)

# 2. Explode the list into new rows 
# (Pandas automatically converts list items back into plain strings here!)
df = df.explode('discount_split')

# 3. Clean up: Replace the old messy column with the clean one
df['discount'] = df['discount_split'].fillna(df['discount'])
df = df.drop(columns=['discount_split'])

# 4. Reset the index so your row numbers (9, 10, 11...) stay sequential
df = df.reset_index(drop=True)

In [120]:
# Macthing discount types: voor 0.99, discounted_price - 0.99, discount_type = fixed_discount, price before- NA, product_quantity = 1
#                          2e gratis, discounted price - NA, discount_type = buy_x_get_y, product_quantity = 2
#                          1euro korting, discounted_price - NA, discount_type = fixed_korting_money, product quantity = 1
#                          50% korting, discounted_price - NA, discount_type = fixed_korting, product_quantity = 1
#                          per x grams/ml y - discounted_price = 1.70, discount_type = fixed_discount, price before - NA, product_quantity = 100 gram/ml
#                          where we have van x voor y - discounted_price = y, orignal_price = x, 

In [121]:
def extract_prices(discount):
    if not isinstance(discount, str):
        return pd.Series({"discount": discount, "original_price": None, "discounted_price": None})
    
    parts = discount.split(",")
    
    if len(parts) == 1:
        voor_match = re.search(r'voor (\d+\.?\d*)', discount)
        discounted_price = float(voor_match.group(1)) if voor_match else None
        return pd.Series({
            "discount": discount.strip(),
            "original_price": None,
            "discounted_price": discounted_price
        })
    
    else:
        discount_label = parts[0].strip()
        van_match = re.search(r'van (\d+\.?\d*)', discount)
        voor_match = re.search(r'voor (\d+\.?\d*)', discount)
        original_price = float(van_match.group(1)) if van_match else None
        discounted_price = float(voor_match.group(1)) if voor_match else None
        return pd.Series({
            "discount": discount_label,
            "original_price": original_price,
            "discounted_price": discounted_price
        })

def normalize_discount(discount):
    if not isinstance(discount, str):
        return pd.Series({"discount_type": "unknown", "quantity": 1, "discount_value": None})
    
    d = discount.strip().lower()
    d = re.sub(r'^uitgelicht\s+', '', d)
    
    # "voor 0.99" — fixed price
    if re.match(r'^voor \d+\.?\d*$', d):
        return pd.Series({"discount_type": "fixed_price", "quantity": 1, "discount_value": None})

    # "2e gratis" — buy 2 get 1 free = 50% overall
    if re.match(r'^\de gratis', d):
        qty = int(re.match(r'^(\d)e gratis', d).group(1))
        # e.g. "2e gratis" = buy 2 pay for 1 = 50% overall
        # "3e gratis" = buy 3 pay for 2 = 33.3% overall
        overall_pct = round((1 / qty) * 100, 1)
        return pd.Series({"discount_type": "buy_x_get_y_free", "quantity": qty, "discount_value": overall_pct})

    # "1+1 gratis" or "2+2 gratis"
    if re.match(r'^\d\+\d gratis', d):
        match = re.match(r'^(\d)\+(\d) gratis', d)
        buy = int(match.group(1))
        free = int(match.group(2))
        total = buy + free
        # e.g. 1+1 = pay for 1 get 2 = 50% overall
        # e.g. 2+2 = pay for 2 get 4 = 50% overall
        overall_pct = round((free / total) * 100, 1)
        return pd.Series({"discount_type": "buy_x_get_y_free", "quantity": total, "discount_value": overall_pct})

    # "50% korting" — percentage discount, already overall
    if re.search(r'\d+% korting', d):
        pct = float(re.search(r'(\d+\.?\d*)%', d).group(1))
        return pd.Series({"discount_type": "percentage_discount", "quantity": 1, "discount_value": pct})

    # "€1 korting" — fixed euro discount
    if re.search(r'€\d+\.?\d*\s*korting', d):
        amount = float(re.search(r'€(\d+\.?\d*)', d).group(1))
        return pd.Series({"discount_type": "fixed_euro_discount", "quantity": 1, "discount_value": amount})

    # "2 voor 3.99" — multi buy, calculate % saving if we have original price
    if re.match(r'^\d+ stuks? \d+\.?\d*$', d):
        match = re.match(r'^(\d+) stuks? (\d+\.?\d*)$', d)
        qty = int(match.group(1))
        price = float(match.group(2))
        return pd.Series({"discount_type": "multi_buy", "quantity": qty, "discount_value": None})

    # "2e halve prijs" — 2nd item half price = 25% overall
    if re.search(r'\de halve prijs', d):
        return pd.Series({"discount_type": "second_half_price", "quantity": 2, "discount_value": 25.0})

    # "2 stuks 40%" — quantity based percentage
    if re.match(r'^\d+ stuk', d):
        match = re.match(r'^(\d+) stuk', d)
        pct_match = re.search(r'(\d+)%', d)
        qty = int(match.group(1))
        pct = float(pct_match.group(1)) if pct_match else None
        # pct applies to ALL items, not split across them
        return pd.Series({"discount_type": "quantity_percentage", "quantity": qty, "discount_value": pct})

    return pd.Series({"discount_type": "other", "quantity": 1, "discount_value": None})

# Step 1: Extract prices and clean discount label
df[["discount", "original_price", "discounted_price"]] = df["discount"].apply(extract_prices)

# Step 2: Normalize discount type
normalized = df["discount"].apply(normalize_discount)
df = pd.concat([df, normalized], axis=1)

In [122]:
df.loc[df["discount_type"] == "fixed_euro_discount", "discount_value"] = None

In [123]:
df["start_date"] = pd.to_datetime(df["start_date"], unit="ms").dt.date
df["end_date"] = pd.to_datetime(df["end_date"], unit="ms").dt.date

In [124]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(df)

,product,discount,link,store,start_date,end_date,product_info,discounted_price,original_price,discount_type,quantity,discount_value
0,AH Aardappelpartjes Provençaals gekruid 600 gram,voor 0.99,https://www.ah.nl/bonus/groep/766837?week=9,Albertijn,2026-02-23,2026-03-01,,0.99,NaN,fixed_price,1,NaN
1,AH Geschrapte worteltjes 500 gram,2e gratis,https://www.ah.nl/bonus/groep/766809?week=9,Albertijn,2026-02-23,2026-03-01,,NaN,NaN,buy_x_get_y_free,2,50.0
2,AH Rucola 85 gram,voor 0.99,https://www.ah.nl/bonus/groep/766835?week=9,Albertijn,2026-02-23,2026-03-01,,0.99,NaN,fixed_price,1,NaN
3,AH Biologisch winterpeen 750 gram en AH Biolog...,2 voor 1.99,https://www.ah.nl/bonus/groep/775045?week=9,Albertijn,2026-02-23,2026-03-01,,1.99,NaN,other,1,NaN
4,Alle AH Italiaanse verspakketten,€1 korting,https://www.ah.nl/bonus/groep/766864?week=9,Albertijn,2026-02-23,2026-03-01,,NaN,NaN,fixed_euro_discount,1,NaN
5,AH Vastkokende aardappelen 3 kilo,2e gratis,https://www.ah.nl/bonus/groep/775044?week=9,Albertijn,2026-02-23,2026-03-01,,NaN,NaN,buy_x_get_y_free,2,50.0
6,AH Andijvie fijngesneden 500 gram,50% korting,https://www.ah.nl/bonus/groep/766745?week=9,Albertijn,2026-02-23,2026-03-01,,NaN,NaN,percentage_discount,1,50.0
7,AH Mexicaanse roerbakgroenten 400 gram,2 voor 3.99,https://www.ah.nl/bonus/groep/766850?week=9,Albertijn,2026-02-23,2026-03-01,,3.99,NaN,other,1,NaN
8,AH Snoepgroente tomaat 1 kilo,25% korting,https://www.ah.nl/bonus/groep/775043?week=9,Albertijn,2026-02-23,2026-03-01,,NaN,NaN,percentage_discount,1,25.0
9,AH Verspakket rode linzensoep,€1 korting,https://www.ah.nl/bonus/groep/766763?week=9,Albertijn,2026-02-23,2026-03-01,,NaN,NaN,fixed_euro_discount,1,NaN


In [125]:
df.to_json("Albertijn_deals.json", orient="records", force_ascii=False, indent=2)